In [1]:
from clean_file import run_cleaning_pipeline
from preprocessing_file import run_preprocessing_pipeline

In [2]:
import pandas as pd

df_raw = pd.read_csv('../data/house_price.csv')
print('Raw data shape:', df_raw.shape)
df_raw.head()

Raw data shape: (13320, 9)


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
df_clean = run_cleaning_pipeline(df_raw)
X, y = run_preprocessing_pipeline(df_clean.copy())

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (13262, 246)
y shape: (13262,)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import warnings
warnings.filterwarnings('ignore')

models = {
    "LinearRegression": LinearRegression(),
    "Ridge":            Ridge(),
    "Lasso":            Lasso(),
    "DecisionTree":     DecisionTreeRegressor(random_state=42),
    "RandomForest":     RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

cv_results = {}

for name, model in models.items():
    print(f"Training {name}...")
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    cv_results[name] = scores.mean()

sorted_results = sorted(cv_results.items(), key=lambda x: x[1], reverse=True)

print("\nCross-Validation R\u00b2 Scores:")
print("-" * 35)
for name, score in sorted_results:
    print(f"{name:<22}: {score:.4f}")

print(f"\n best model: {sorted_results[0][0]}  |  R square = {sorted_results[0][1]:.4f}")

Training LinearRegression...
Training Ridge...
Training Lasso...
Training DecisionTree...
Training RandomForest...
Training GradientBoosting...

Cross-Validation R² Scores:
-----------------------------------
RandomForest          : 0.7862
GradientBoosting      : 0.7702
Ridge                 : 0.7215
LinearRegression      : 0.7212
DecisionTree          : 0.6716
Lasso                 : -0.0003

أفضل موديل: RandomForest  |  R² = 0.7862


In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20],
    'min_samples_split': [2, 5],
    'max_features':      ['sqrt', 0.5]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("best Hyperparameters:", grid_search.best_params_)
print("best R square CV: ,  {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 16 candidates, totalling 48 fits
أفضل Hyperparameters: {'max_depth': 20, 'max_features': 0.5, 'min_samples_split': 5, 'n_estimators': 200}
أفضل R² على CV:       0.7850


In [9]:
from sklearn.metrics import r2_score

best_rf = grid_search.best_estimator_

y_pred_train = best_rf.predict(X_train)
y_pred_test  = best_rf.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test,  y_pred_test)

print(f"R square  Train Set: {r2_train:.4f}")
print(f"R square Test Set:  {r2_test:.4f}")
print()



R square  Train Set: 0.9059
R square Test Set:  0.7947

